# 06 — CSP target assignment

Each target is a variable and each drone is a possible value. A drone can take a target only when the package still fits in its capacity and the estimated out-and-back energy still fits in `battery * (1 - reserve)`. MRV picks the target with the fewest remaining drones. Forward checking stops as soon as another target is left with none.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
from uavplanner.config import Config
from uavplanner.csp import avg_cell_cost, csp_assign, estimate_target_cost
from uavplanner.models import Drone, EnergyCoef
from uavplanner.viz import write_figures_4
from uavplanner.world import make_targets, make_world

cfg = Config()
rng = np.random.default_rng(cfg.seed)
world = make_world(cfg, rng)
targets = make_targets(world, cfg, rng)
drones = [Drone(id=i, battery=cfg.battery, capacity_kg=cfg.capacity_kg) for i in (1, 2, 3)]
average = avg_cell_cost(world, EnergyCoef(), rng, samples=cfg.csp_samples)
estimates = {target.id: estimate_target_cost(world.base, target, average) for target in targets}
plain, plain_stats = csp_assign(targets, drones, estimates, reserve=cfg.reserve, use_mrv=False, use_fc=False)
smart, smart_stats = csp_assign(targets, drones, estimates, reserve=cfg.reserve)
print(f'targets: {len(targets)}; mean cell cost: {average:.3f} EU')
print(f'plain backtracks: {plain_stats["backtracks"]}; MRV+FC backtracks: {smart_stats["backtracks"]}')
print('assignment (target -> drone):', smart)


In [ ]:
figure = write_figures_4(ROOT / 'results')
print(f'Wrote {figure}')
